<a href="https://colab.research.google.com/github/datawrangler7798/Vector-Databases-Hands-On/blob/main/Pinecone_Vector_DB_Production_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Pinecone Vector Databases** — DilSeVector, Production Edition 💘
The code a real 2026 engineering team actually ships. Text in → matches out. You will never touch a vector today.

The rule of this notebook: every fix is small. Watch the line counts:
🎫#2 exact facts → 1 line (a metadata filter) + a sparse index for keyword relevance
🎫#3 rank-12 match → 4 lines (a rerank= parameter on the SAME search call)
🎫#1 300 GB RAM → 0 lines (managed compression — we do the math, Pinecone does the work)
🎫#4 launch night → ~10 lines (a timer + a cache dict; the real dashboard is the console)
Before class: run Cells 0–3 (index creation takes ~1 min, upsert+embedding ~1 min). Everything else is live. During class: every 🎲 CHAT BET gets pasted into Zoom chat BEFORE running the cell. Never run in silence.

In [ ]:
!pip -q install pinecone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 4.4 MB/s eta 0:00:00


In [ ]:
from pinecone import Pinecone
import time

pc = Pinecone(api_key="YOUR API KEY")   # from app.pinecone.io
print("Connected ✅")

Connected ✅


#The user base: 400 profiles

In [ ]:
import random
random.seed(42)

LOCALITIES = ["Dadar","Thane","Byculla","Dombivli","South Bombay","Bandra","Andheri","Powai"]
COLLEGES   = ["IIT Bombay","VJTI","NMIMS","St Xaviers","Mumbai University","SPIT","ICT Mumbai"]
JOBS       = ["software engineer at TCS","chartered accountant","doctor","startup founder",
              "graphic designer","data scientist","teacher","investment banker","architect"]
VIBES = ["adventurous and loves trekking in the Sahyadris",
         "fun-loving but settled, enjoys quiet weekends",
         "foodie who hunts for the best vada pav in the city",
         "bookworm, poetry evenings at Prithvi cafe",
         "gym at 6am, marathon runner, discipline is life",
         "stand-up comedy fan, drags friends to open mics",
         "loves dogs, volunteers at animal shelters on weekends",
         "sorted but chill, big on family, small on drama",
         "travel junkie, 14 countries and counting",
         "classical music and long walks at Marine Drive"]
LANGS = ["Marathi","Gujarati","Hindi","Konkani","Tamil"]

users = []
for i in range(400):
    u = {"_id": f"P{i:04d}", "locality": random.choice(LOCALITIES), "college": random.choice(COLLEGES),
         "batch": random.choice(range(2014, 2023)), "job": random.choice(JOBS),
         "vibe": random.choice(VIBES), "lang": random.choice(LANGS),
         "smoke": random.choice(["non-smoker", "occasional smoker"])}
    users.append(u)

# planted demo profiles
users[1].update(college="IIT Bombay", batch=2019, lang="Marathi", locality="Dadar", smoke="non-smoker",
                vibe="loves dogs, volunteers at animal shelters on weekends", job="data scientist")
users[2].update(vibe="sorted but chill, big on family, small on drama", locality="Bandra")
users[3].update(smoke="non-smoker", locality="Dadar", job="architect",
                vibe="loves dogs, has a golden retriever named Simba")

for u in users:   # the bio is what the app shows on a profile card
    u["bio"] = (f"{u['job'].capitalize()}, {u['college']} {u['batch']} batch. {u['lang']} speaking, "
                f"lives in {u['locality']}. {u['vibe'].capitalize()}. {u['smoke'].capitalize()}.")

for i in (1, 2, 3): print(users[i]["_id"], "→", users[i]["bio"])
print("Total users:", len(users))

P0001 → Data scientist, IIT Bombay 2019 batch. Marathi speaking, lives in Dadar. Loves dogs, volunteers at animal shelters on weekends. Non-smoker.
P0002 → Architect, Mumbai University 2014 batch. Tamil speaking, lives in Bandra. Sorted but chill, big on family, small on drama. Occasional smoker.
P0003 → Architect, St Xaviers 2018 batch. Konkani speaking, lives in Dadar. Loves dogs, has a golden retriever named simba. Non-smoker.
Total users: 400


#Create the index (this is the whole "embedding pipeline")
Say: "Notice what's NOT here: no model download, no .encode(), no numpy. We tell Pinecone which hosted embedding model to use and WHICH FIELD to embed. Done. This is how production teams ship."

In [ ]:
DENSE = "dilse-dense"

if not pc.has_index(DENSE):
    pc.create_index_for_model(
        name=DENSE, cloud="aws", region="us-east-1",
        embed={"model": "llama-text-embed-v2",     # Pinecone-hosted embedder (the Romantic)
               "field_map": {"text": "bio"}}       # ← embed the 'bio' field of each record
    )
while not pc.describe_index(DENSE).status["ready"]: time.sleep(1)
dense = pc.Index(DENSE)
print("Dense index ready ✅")

Dense index ready ✅


#Upsert: send TEXT, get vectors (you never see them)
Records go up as plain dicts. Pinecone embeds bio server-side; the other fields become filterable metadata. (96 records per call is the integrated-embedding limit — hence the tiny batching loop. Batching lesson #1, for free.)

In [ ]:
NS = "users"

for i in range(0, len(users), 96):
    dense.upsert_records(
        namespace=NS,
        records=users[i:i+96]
    )

time.sleep(10)

print(dense.describe_index_stats())

DescribeIndexStatsResponse(dimension=1024, total_vector_count=400, metric='cosine', namespaces=1)


# Search = 5 lines. The Romantic at her best.
🎲 CHAT BET: "Query: 'someone adventurous who loves trekking'. Will the top hit contain the word 'adventurous'? YES/NO." (Either answer teaches: matches come by MEANING — 'travel junkie, 14 countries' can win without the word.)

In [ ]:
def search(index, text, k=5, flt=None):

    q = {
        "inputs": {"text": text},
        "top_k": k
    }

    if flt:
        q["filter"] = flt

    res = index.search(
        namespace=NS,
        query=q,
        fields=["bio"]
    )

    return [
        (h.id, round(h.score, 4), h.fields["bio"])
        for h in res.result.hits
    ]


def show(hits, star=()):

    for r, (pid, s, bio) in enumerate(hits, 1):

        print(
            f"{r:>2}. {pid}  {s}"
            f"{'  ⭐' if pid in star else ''}\n"
            f"    {bio[:105]}"
        )


show(search(dense, "someone adventurous who loves trekking"))

 1. P0187  0.3696
    Architect, ICT Mumbai 2022 batch. Tamil speaking, lives in Andheri. Adventurous and loves trekking in the
 2. P0220  0.3566
    Architect, Mumbai University 2022 batch. Hindi speaking, lives in Thane. Adventurous and loves trekking i
 3. P0093  0.3558
    Data scientist, IIT Bombay 2022 batch. Konkani speaking, lives in Bandra. Adventurous and loves trekking 
 4. P0347  0.3555
    Doctor, IIT Bombay 2017 batch. Hindi speaking, lives in Andheri. Adventurous and loves trekking in the sa
 5. P0195  0.3527
    Software engineer at tcs, VJTI 2020 batch. Gujarati speaking, lives in Dadar. Adventurous and loves trekk


#🎫 CRISIS #2, live: the Romantic meets a biodata query
🎲 CHAT BET: "'IIT Bombay 2019 batch, Marathi, lives in Dadar' — of the top 5, how many match ALL FOUR facts? Type 0–5." The ✓/✗ table does the arguing. Expected: near-misses (right college, wrong batch...). The Romantic hears vibes, not facts.

[ ]


In [ ]:
CRISIS2 = "IIT Bombay 2019 batch, Marathi speaking, lives in Dadar"

def fact_check(hits):
    print(f"{'rank':<5}{'id':<7}{'IIT-B':<7}{'2019':<7}{'Marathi':<9}{'Dadar':<7}")
    for r, (pid, s, bio) in enumerate(hits, 1):
        f = ["✓" if w in bio else "✗" for w in ("IIT Bombay", "2019", "Marathi", "Dadar")]
        print(f"{r:<5}{pid:<7}{f[0]:<7}{f[1]:<7}{f[2]:<9}{f[3]:<7}{' ⭐ PERFECT' if f==['✓']*4 else ''}")

fact_check(search(dense, CRISIS2))
print("\n(The perfect profile P0001 exists. Did the Romantic even surface it?)")

rank id     IIT-B  2019   Marathi  Dadar  
1    P0388  ✗      ✓      ✓        ✗      
2    P0001  ✓      ✓      ✓        ✓       ⭐ PERFECT
3    P0157  ✗      ✓      ✓        ✗      
4    P0177  ✓      ✓      ✗        ✗      
5    P0218  ✓      ✓      ✗        ✗      

(The perfect profile P0001 exists. Did the Romantic even surface it?)


#Production fix #1: facts don't belong in embeddings. They belong in FILTERS.
ONE LINE. This is the single most production-real lesson of the day: structured facts → metadata filter; the embedding only handles the fuzzy part.
🎲 CHAT BET: "Same search + a metadata filter. Perfect matches in the top 5: how many now?"

In [ ]:
hits = search(dense, "looking for a genuine connection",         # the fuzzy part
              flt={"college": "IIT Bombay", "batch": 2019,        # ← the facts, as a WHERE clause
                   "lang": "Marathi", "locality": "Dadar"})
fact_check(hits)
print("\nRemember Session 3's trick question — 'similar AND filtered'? BOTH, in one call. That was the answer.")

rank id     IIT-B  2019   Marathi  Dadar  
1    P0001  ✓      ✓      ✓        ✓       ⭐ PERFECT

Remember Session 3's trick question — 'similar AND filtered'? BOTH, in one call. That was the answer.


# But what about keyword relevance? Meet the Aunty (a real sparse index)
Filters are binary: in or out. Sometimes you want exact words to boost ranking, not gatekeep — "golden retriever" should RANK higher, not exclude everyone else. That's a sparse index: Pinecone hosts a BM25-family model (pinecone-sparse-english-v0). Same API, second index. Zero math.

In [ ]:
SPARSE = "dilse-sparse"

if not pc.has_index(SPARSE):

    pc.create_index_for_model(
        name=SPARSE,
        cloud="aws",
        region="us-east-1",
        embed={
            "model": "pinecone-sparse-english-v0",
            "field_map": {"text": "bio"}
        }
    )

while not pc.describe_index(SPARSE).status["ready"]:
    time.sleep(1)

sparse = pc.Index(SPARSE)

for i in range(0, len(users), 96):
    sparse.upsert_records(
        namespace=NS,
        records=users[i:i+96]
    )

time.sleep(10)

print("Sparse index ready ✅")

Sparse index ready ✅


#The Aunty's superpower... and her blind spot, back-to-back
🎲 CHAT BET 1: "Aunty on the biodata query — perfect match at #1: YES/NO?" 🎲 CHAT BET 2: "Query 'fun-loving but settled'. P0002 says 'sorted but chill' — ZERO shared words. Will Aunty find P0002 in top 5? Will the Romantic?"


In [ ]:
print("=== AUNTY (sparse) on the biodata query ===")
fact_check(search(sparse, CRISIS2))

print("\n=== AUNTY on: 'fun-loving but settled' ===")
show(search(sparse, "fun-loving but settled, enjoys a quiet weekend"), star={"P0002"})

print("\n=== ROMANTIC (dense) on the same query ===")
show(search(dense, "fun-loving but settled, enjoys a quiet weekend"), star={"P0002"})
print("\nComplementary blind spots → hire both. That's hybrid.")

=== AUNTY (sparse) on the biodata query ===
rank id     IIT-B  2019   Marathi  Dadar  
1    P0001  ✓      ✓      ✓        ✓       ⭐ PERFECT
2    P0050  ✓      ✗      ✓        ✓      
3    P0166  ✓      ✓      ✓        ✗      
4    P0107  ✓      ✗      ✓        ✗      
5    P0374  ✓      ✗      ✓        ✗      

=== AUNTY on: 'fun-loving but settled' ===
 1. P0299  12.2463
    Graphic designer, NMIMS 2014 batch. Hindi speaking, lives in Byculla. Fun-loving but settled, enjoys quie
 2. P0255  12.0908
    Architect, Mumbai University 2020 batch. Hindi speaking, lives in Powai. Fun-loving but settled, enjoys q
 3. P0164  12.0863
    Teacher, Mumbai University 2016 batch. Konkani speaking, lives in Andheri. Fun-loving but settled, enjoys
 4. P0226  12.0694
    Architect, Mumbai University 2015 batch. Hindi speaking, lives in Bandra. Fun-loving but settled, enjoys 
 5. P0173  12.0547
    Architect, VJTI 2020 batch. Marathi speaking, lives in Dadar. Fun-loving but settled, enjoys quiet weeken

#Hybrid: query both, merge with RRF (the pattern production actually uses)
Two indexes, one merge function. Reciprocal Rank Fusion — trust ranks, not raw scores (which live on different scales). α still exists: it weights how much you trust each matchmaker.
🎲 CHAT GAME (α auction): "Type ONE number 0–1: your α for a matrimony app. Now for casual dating." Run with the room's median, then re-run at 0.9 and 0.1 — predict what breaks each time.

In [ ]:
def hybrid(text, alpha=0.5, k=5):
    d = [pid for pid, _, _ in search(dense,  text, k=25)]
    s = [pid for pid, _, _ in search(sparse, text, k=25)]
    score = {}
    for rank, pid in enumerate(d): score[pid] = score.get(pid, 0) + alpha       / (60 + rank)
    for rank, pid in enumerate(s): score[pid] = score.get(pid, 0) + (1 - alpha) / (60 + rank)
    top = sorted(score, key=score.get, reverse=True)[:k]
    bio = {u["_id"]: u["bio"] for u in users}
    return [(pid, round(score[pid], 5), bio[pid]) for pid in top]

MIXED = "IIT Bombay 2019 Marathi Dadar, adventurous trekking type"   # facts AND vibes in one query
for a in (0.0, 0.5, 1.0):
    print(f"\n──── α = {a}  ({'pure AUNTY' if a==0 else 'pure ROMANTIC' if a==1 else 'diplomat'}) ────")
    show(hybrid(MIXED, alpha=a, k=3), star={"P0001"})


──── α = 0.0  (pure AUNTY) ────
 1. P0001  0.01667  ⭐
    Data scientist, IIT Bombay 2019 batch. Marathi speaking, lives in Dadar. Loves dogs, volunteers at animal
 2. P0050  0.01639
    Doctor, IIT Bombay 2015 batch. Marathi speaking, lives in Dadar. Travel junkie, 14 countries and counting
 3. P0166  0.01613
    Architect, IIT Bombay 2019 batch. Marathi speaking, lives in Bandra. Stand-up comedy fan, drags friends t

──── α = 0.5  (diplomat) ────
 1. P0001  0.01591  ⭐
    Data scientist, IIT Bombay 2019 batch. Marathi speaking, lives in Dadar. Loves dogs, volunteers at animal
 2. P0093  0.01505
    Data scientist, IIT Bombay 2022 batch. Konkani speaking, lives in Bandra. Adventurous and loves trekking 
 3. P0387  0.01317
    Architect, ICT Mumbai 2019 batch. Marathi speaking, lives in Dadar. Classical music and long walks at mar

──── α = 1.0  (pure ROMANTIC) ────
 1. P0005  0.01667
    Graphic designer, NMIMS 2019 batch. Konkani speaking, lives in Thane. Adventurous and loves trekk

# leave on screen: "After break: the investor's 300 GB email — and why the fix is ZERO lines of your code."
#🎫 CRISIS #1: The Burn-Rate math (do it WITH the class in chat)
Then the punchline: quantization. On screen next: the by-hand chat race — the only 'code' is the one your BRAIN runs.


In [ ]:
N, dim, fbytes = 100_000_000, 1024, 4        # llama-text-embed-v2 default: 1024 dims
print(f"{N:,} users × {dim} dims × {fbytes} B = {N*dim*fbytes/1e9:,.0f} GB of RAM. Investor: 😡")
print(f"After ~32× product quantization: {N*dim*fbytes/1e9/32:,.0f} GB. Investor: 😍")

100,000,000 users × 1024 dims × 4 B = 410 GB of RAM. Investor: 😡
After ~32× product quantization: 13 GB. Investor: 😍


#⭐ PQ by hand (the chat race — this IS the lesson)
Paste in chat: "Compress this person! For each chunk pick the closest TYPE (0–3). Type your 4 codes, e.g. 0 3 1 2. GO!"


PROFILE VECTOR (8 numbers, chunked in 2s):
    [ 0.9, 0.1 | 0.8, 0.2 | 0.1, 0.9 | 0.5, 0.5 ]

CODEBOOK OF TYPES (valid for every chunk):
    Type 0: (0.9, 0.1)    Type 1: (0.1, 0.9)
    Type 2: (0.5, 0.5)    Type 3: (0.8, 0.3)
Reveal: 0 3 1 2 · 32 bytes → 1 byte = 32× · decode chunk 2: you stored (0.8, 0.2) as (0.8, 0.3) — that gap is quantization error. Aunty remembers you as your type, not as YOU. Same devil as yesterday's ANN deal — new contract.
Then the production truth (say it slowly): "And here is the best part: in Pinecone, YOU WRITE NONE OF THIS. Compression, codebooks, retraining when the user base drifts — managed, under the hood. The knobs you learned (m = chunks, nbits = types-per-chunk) are what you'd tune in FAISS if you ever build the engine yourself — and what you now READ fluently when Pinecone's docs mention them."

In [ ]:
# Optional under-the-hood peek (FAISS) — for the curious, NOT for production. Flip to run.
SHOW_ENGINE_ROOM = False
if SHOW_ENGINE_ROOM:
    # !pip -q install faiss-cpu numpy
    import faiss, numpy as np
    X = np.random.rand(100_000, 128).astype("float32")
    pq = faiss.IndexPQ(128, 16, 8)          # 16 chunks, 2^8 = 256 types per chunk
    pq.train(X); pq.add(X)
    print(f"raw: {X.nbytes/1e6:.0f} MB  →  PQ codes: {100_000*16/1e6:.1f} MB  (32×)")

#🎫 CRISIS #3a: prove matching is good. With numbers.
These five tiny functions ARE what practitioners write (or import from an evals library). First the chat race on the card, then the same metrics on OUR app — since we generated the users, we KNOW who's truly compatible (real teams buy this ground truth with human labels).
🎲 CHAT RACE: "Shown [A,B,C,D,E], compatible {B,E,G}. Compute P@5, R@5, MRR@5 — fastest correct wins."

In [ ]:
import math
def precision_at_k(ranked, rel, k): return sum(x in rel for x in ranked[:k]) / k
def recall_at_k(ranked, rel, k):    return sum(x in rel for x in ranked[:k]) / max(1, len(rel))
def mrr_at_k(ranked, rel, k):
    return next((1/i for i, x in enumerate(ranked[:k], 1) if x in rel), 0.0)
def ap_at_k(ranked, rel, k):
    hits = ap = 0
    for i, x in enumerate(ranked[:k], 1):
        if x in rel: hits += 1; ap += hits / i
    return ap / max(1, len(rel))
def ndcg_at_k(ranked, gains, k):
    dcg  = sum(gains.get(x, 0) / math.log2(i+1) for i, x in enumerate(ranked[:k], 1))
    idcg = sum(g / math.log2(i+1) for i, g in enumerate(sorted(gains.values(), reverse=True)[:k], 1))
    return dcg / idcg if idcg else 0.0

card_ranked, card_rel = ["A","B","C","D","E"], {"B","E","G"}
print(f"CARD → P@5={precision_at_k(card_ranked, card_rel, 5):.2f}  R@5={recall_at_k(card_ranked, card_rel, 5):.2f}  "
      f"MRR@5={mrr_at_k(card_ranked, card_rel, 5):.2f}  AP@5={ap_at_k(card_ranked, card_rel, 5):.2f}")
print("Why does AP divide by 3, not 2? The match we FAILED to show (G) must hurt the score.")

CARD → P@5=0.40  R@5=0.67  MRR@5=0.50  AP@5=0.30
Why does AP divide by 3, not 2? The match we FAILED to show (G) must hurt the score.


#The metrics on OUR app
🎲 CHAT BET: "Query persona: non-smoker · loves dogs · Dadar. Guess the rank of the FIRST truly compatible profile in plain dense top-20." Talking point on output: decent recall, painful MRR = Crisis #3, measured. "How far does she scroll before the first real match? MRR is the patience-meter."

In [ ]:
QUERY3 = "non-smoker who loves dogs, based in Dadar"
truly = {u["_id"] for u in users
         if u["smoke"] == "non-smoker" and "dog" in u["vibe"] and u["locality"] == "Dadar"}
print(f"Ground truth: {len(truly)} truly compatible → {sorted(truly)}\n")

stage1 = [pid for pid, _, _ in search(dense, QUERY3, k=20)]
first = next((i for i, x in enumerate(stage1, 1) if x in truly), ">20")
print("Dense top-20:", stage1[:10], "…")
print(f"P@5={precision_at_k(stage1, truly, 5):.2f}  R@10={recall_at_k(stage1, truly, 10):.2f}  "
      f"MRR@10={mrr_at_k(stage1, truly, 10):.2f}  |  first TRUE match at rank {first}")

Ground truth: 4 truly compatible → ['P0001', 'P0003', 'P0011', 'P0145']

Dense top-20: ['P0011', 'P0280', 'P0064', 'P0145', 'P0085', 'P0213', 'P0250', 'P0053', 'P0382', 'P0235'] …
P@5=0.40  R@10=0.50  MRR@10=1.00  |  first TRUE match at rank 1


#BREAK 2 · teaser: "The rank-12 fix is four lines. Not four hundred. Four."
Cell 9 — 🎫 CRISIS #3b: Reranking = ONE parameter on the SAME call
Say: "Stage 1 swipes (fast, 400 → 25 maybes). Stage 2 is the coffee date: a cross-encoder reads the query and each profile TOGETHER — like your brain did when you insta-picked the golden-retriever profile. In production, that judge is hosted. You add a rerank block. That's the entire diff."
🎲 CHAT BET: "After reranking: MRR@10 UP/DOWN/SAME? P@5?"

In [ ]:
es = dense.search(
    namespace=NS,
    query={
        "inputs": {"text": QUERY3},
        "top_k": 25
    },
    fields=["bio"],
    rerank={
        "model": "bge-reranker-v2-m3",
        "top_n": 10,
        "rank_fields": ["bio"]
    }
)

stage2 = [h.id for h in es.result.hits]

print(f"{'metric':<8}{'before':>9}{'after':>9}")

for name, fn, k in [
    ("MRR@10", mrr_at_k, 10),
    ("P@5", precision_at_k, 5),
    ("AP@10", ap_at_k, 10)
]:
    print(
        f"{name:<8}"
        f"{fn(stage1, truly, k):>9.2f}"
        f"{fn(stage2, truly, k):>9.2f}"
    )

print("\nRank 12 → top 3. Prasad's cousin gets her match. 🎫#3 CLOSED ✅")

print(
    "Anti-confusion: HYBRID blends scores AT retrieval. "
    "RERANK is a second pass AFTER. Party vs shortlist."
)

metric     before    after
MRR@10       1.00     1.00
P@5          0.40     0.80
AP@10        0.38     1.00

Rank 12 → top 3. Prasad's cousin gets her match. 🎫#3 CLOSED ✅
Anti-confusion: HYBRID blends scores AT retrieval. RERANK is a second pass AFTER. Party vs shortlist.


# 🎫 CRISIS #4: Launch night. Latency percentiles in 10 lines.
First show the incident table (below) and let the room diagnose in chat. THEN run the timer on our real index.
QPS	p50	p95	p99
150,000	5 ms	50 ms	1,200 ms
🎲 CHAT: "Prasad says '150K QPS is AMAZING!' What do you tell him? Launch-ready: YES/NO + why." Land on: p99 = the 1% staring at a spinner for 1.2 s mid-swipe ≈ 1,500 ruined moments/second at that QPS. Averages flirt; tails tell the truth. (Note: our numbers below include network to the cloud — that's honest end-to-end latency, the number users feel.)

In [ ]:
import numpy as np
lat = []
for _ in range(60):
    t0 = time.perf_counter()
    search(dense, "dog lover in Dadar", k=10)
    lat.append((time.perf_counter() - t0) * 1000)

p50, p95, p99 = np.percentile(lat, [50, 95, 99])
print(f"end-to-end latency  p50={p50:.0f} ms   p95={p95:.0f} ms   p99={p99:.0f} ms")
print("\nIn production you don't hand-roll this: the Pinecone console charts QPS + percentiles live,")
print("and teams pipe them to Prometheus/Datadog. [→ 60-second console tour now]")

end-to-end latency  p50=109 ms   p95=183 ms   p99=467 ms

In production you don't hand-roll this: the Pinecone console charts QPS + percentiles live,
and teams pipe them to Prometheus/Datadog. [→ 60-second console tour now]


#The free lunch: caching (8 lines, and yes, real apps do exactly this)
🎲 CHAT BET: "'matches near Dadar' is asked 5,000×/day. Cache hit vs fresh search — how many × faster? 5 / 50 / 500?" Gotchas to name after: invalidation (new users join → stale cache) and near-duplicates ('matches near Dadar' vs 'Dadar-area matches'). In prod: same dict, but it's called Redis.

In [ ]:
cache = {}
def cached_search(text, k=5):
    key = text.strip().lower()
    if key not in cache:
        cache[key] = search(dense, text, k)
    return cache[key]

t0 = time.perf_counter(); cached_search("matches near Dadar"); miss = (time.perf_counter()-t0)*1000
t0 = time.perf_counter(); cached_search("matches near Dadar"); hit  = (time.perf_counter()-t0)*1000
print(f"MISS {miss:8.2f} ms  (embed + search, over the network)")
print(f"HIT  {hit:8.4f} ms  →  {miss/max(hit, 1e-9):,.0f}× faster. 🎫#4 CLOSED ✅")

MISS   298.55 ms  (embed + search, over the network)
HIT    0.0943 ms  →  3,164× faster. 🎫#4 CLOSED ✅


The Close (no code)
Point at the four crises in chat; the class closes each in ONE word: #1 quantization (0 lines — managed) · #2 metadata filter + sparse (1 line) · #3 rerank (4 lines) · #4 percentiles + caching (~10 lines).
"Yesterday the algorithm found ONE person a match at a pink café in Dadar. Today it does it for 100 million — affordably, exactly, provably, fast even for the unluckiest 1%... in about 25 lines of application code. That's the difference between a love story and a company. And THAT is why managed vector databases exist."
Homework: invent your own query persona + ground-truth rule; report P@5 and MRR@10 before vs after rerank=. One screenshot, two numbers. Cleanup (avoid charges): pc.delete_index("dilse-dense"); pc.delete_index("dilse-sparse")